# ShopEase - SQL Data Analysis
> CEI Internship | SQL Task 2 | Week 2

This notebook analyzes the ShopEase e-commerce database using Python and SQLite.
We explore customers, products, orders, and generate business insights.

## Step 1: Setup & Imports

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## Step 2: Create Database & Tables

In [ ]:
conn = sqlite3.connect('shopease.db')
cursor = conn.cursor()

# Create tables
cursor.executescript('''
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    email       TEXT UNIQUE NOT NULL,
    city        TEXT NOT NULL,
    state       TEXT NOT NULL,
    join_date   TEXT NOT NULL,
    is_premium  INTEGER DEFAULT 0
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,
    brand        TEXT NOT NULL,
    unit_price   REAL NOT NULL,
    stock_qty    INTEGER NOT NULL DEFAULT 0
);

CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL,
    order_date   TEXT NOT NULL,
    status       TEXT NOT NULL DEFAULT "Pending",
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    item_id      INTEGER PRIMARY KEY,
    order_id     INTEGER NOT NULL,
    product_id   INTEGER NOT NULL,
    quantity     INTEGER NOT NULL,
    unit_price   REAL NOT NULL,
    discount_pct REAL DEFAULT 0,
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
''')

print('Tables created successfully!')

## Step 3: Insert Sample Data

In [ ]:
# Insert Customers
cursor.executemany('INSERT INTO customers VALUES (?,?,?,?,?,?,?,?)', [
    (101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
    (102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
    (103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
    (104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
    (105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
    (106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
    (107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
    (108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0),
])

# Insert Products
cursor.executemany('INSERT INTO products VALUES (?,?,?,?,?,?)', [
    (201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
    (202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
    (203,'Smart Watch','Electronics','Noise',2999.00,150),
    (204,'Running Shoes','Clothing','Nike',4599.00,120),
    (205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
    (206,'Bedsheet Set','Home','Spaces',1299.00,300),
    (207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
    (208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400),
])

# Insert Orders
cursor.executemany('INSERT INTO orders VALUES (?,?,?,?,?)', [
    (1001,101,'2024-08-01','Delivered',4498.00),
    (1002,102,'2024-08-03','Delivered',799.00),
    (1003,103,'2024-08-05','Shipped',7498.00),
    (1004,101,'2024-08-10','Delivered',3499.00),
    (1005,104,'2024-08-12','Cancelled',2999.00),
    (1006,105,'2024-08-15','Delivered',5898.00),
    (1007,106,'2024-08-18','Pending',1299.00),
    (1008,103,'2024-08-20','Delivered',899.00),
    (1009,107,'2024-08-25','Shipped',6098.00),
    (1010,108,'2024-08-28','Delivered',1598.00),
])

# Insert Order Items
cursor.executemany('INSERT INTO order_items VALUES (?,?,?,?,?,?)', [
    (5001,1001,201,2,1499.00,0),
    (5002,1001,207,1,899.00,10),
    (5003,1002,202,1,799.00,0),
    (5004,1003,203,1,2999.00,0),
    (5005,1003,204,1,4599.00,5),
    (5006,1004,205,1,3499.00,0),
    (5007,1005,203,1,2999.00,0),
    (5008,1006,201,1,1499.00,10),
    (5009,1006,204,1,4599.00,5),
    (5010,1007,206,1,1299.00,0),
    (5011,1008,207,1,899.00,0),
    (5012,1009,205,1,3499.00,0),
    (5013,1009,208,2,599.00,15),
    (5014,1010,206,1,1299.00,0),
    (5015,1010,208,1,599.00,0),
])

conn.commit()
print('Data inserted successfully!')

## Step 4: Explore Tables

In [ ]:
# View all customers
df_customers = pd.read_sql_query('SELECT * FROM customers', conn)
print('CUSTOMERS TABLE')
print(f'Total rows: {len(df_customers)}')
df_customers

In [ ]:
# View all products
df_products = pd.read_sql_query('SELECT * FROM products', conn)
print('PRODUCTS TABLE')
print(f'Total rows: {len(df_products)}')
df_products

In [ ]:
# View all orders
df_orders = pd.read_sql_query('SELECT * FROM orders', conn)
print('ORDERS TABLE')
print(f'Total rows: {len(df_orders)}')
df_orders

## Step 5: Business Insights & Visualizations

In [ ]:
# Revenue by Order Status
query = '''
    SELECT status,
           COUNT(*) AS order_count,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM orders
    GROUP BY status
    ORDER BY total_revenue DESC
'''
df_status = pd.read_sql_query(query, conn)
print('Revenue by Order Status:')
print(df_status)

# Plot
colors = ['#2ecc71','#3498db','#e74c3c','#f39c12']
plt.figure(figsize=(7,4))
plt.bar(df_status['status'], df_status['total_revenue'], color=colors)
plt.title('Total Revenue by Order Status')
plt.xlabel('Status')
plt.ylabel('Revenue (Rs.)')
plt.tight_layout()
plt.show()

In [ ]:
# Sales by Product Category
query = '''
    SELECT p.category,
           ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_sales
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_sales DESC
'''
df_cat = pd.read_sql_query(query, conn)
print('Sales by Category:')
print(df_cat)

# Pie Chart
plt.figure(figsize=(6,6))
plt.pie(df_cat['total_sales'], labels=df_cat['category'],
        autopct='%1.1f%%', colors=['#3498db','#e74c3c','#2ecc71'])
plt.title('Sales Distribution by Category')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 Customers by Spending
query = '''
    SELECT c.first_name || ' ' || c.last_name AS customer_name,
           COUNT(o.order_id) AS total_orders,
           ROUND(SUM(o.total_amount), 2) AS total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id
    ORDER BY total_spent DESC
    LIMIT 5
'''
df_top = pd.read_sql_query(query, conn)
print('Top 5 Customers:')
print(df_top)

# Bar Chart
plt.figure(figsize=(8,4))
plt.barh(df_top['customer_name'], df_top['total_spent'], color='#9b59b6')
plt.title('Top 5 Customers by Total Spending')
plt.xlabel('Total Spent (Rs.)')
plt.tight_layout()
plt.show()

In [ ]:
# Average Product Price by Category
query = '''
    SELECT category,
           ROUND(AVG(unit_price), 2) AS avg_price,
           MAX(unit_price) AS max_price,
           MIN(unit_price) AS min_price
    FROM products
    GROUP BY category
'''
df_price = pd.read_sql_query(query, conn)
print('Price Analysis by Category:')
df_price

In [ ]:
# Premium vs Non-Premium Customers
query = '''
    SELECT
        CASE WHEN is_premium = 1 THEN 'Premium' ELSE 'Regular' END AS customer_type,
        COUNT(*) AS count
    FROM customers
    GROUP BY is_premium
'''
df_prem = pd.read_sql_query(query, conn)
print('Customer Type Distribution:')
print(df_prem)

plt.figure(figsize=(5,5))
plt.pie(df_prem['count'], labels=df_prem['customer_type'],
        autopct='%1.1f%%', colors=['#f39c12','#2ecc71'])
plt.title('Premium vs Regular Customers')
plt.tight_layout()
plt.show()

## Step 6: Summary Insights

| Insight | Finding |
|--------|--------|
| Highest revenue status | Delivered orders contribute most |
| Top category | Electronics leads in sales |
| Best customer | Rohan Gupta / Vikram Singh |
| Premium customers | 50% of customer base |
| Most cancelled | 1 order cancelled (Sneha Reddy) |

In [ ]:
# Close connection
conn.close()
print('Analysis complete! Database connection closed.')